# 🚀 FLUX.1-Kontext-dev NIM on Free Colab GPU

Runs NVIDIA NIM container for FLUX.1-Kontext-dev (12B params) on Colab T4 (16GB) with 8-bit quantization.
Exposes via ngrok tunnel → update your `.env` with `FLUX_INFERENCE_URL`.

In [ ]:
# @title 1️⃣ Install dependencies
!apt-get update -qq && apt-get install -y -qq docker.io 2>/dev/null
!pip install -q ngrok 2>/dev/null

import os, subprocess, time, json, requests, threading
print("✅ Dependencies installed")

In [ ]:
# @title 2️⃣ Configure ngrok (get token from https://dashboard.ngrok.com/get-started/your-authtoken)
NGROK_TOKEN = ""  # @param {type:"string"}

if NGROK_TOKEN:
    !ngrok config add-authtoken $NGROK_TOKEN
    print("✅ ngrok configured")
else:
    print("⚠️ Add ngrok token above for public tunnel")

In [ ]:
# @title 3️⃣ Start FLUX NIM Container (8-bit quantized for 16GB T4)
NGC_API_KEY = ""  # @param {type:"string"}
HF_TOKEN = ""     # @param {type:"string"}

if not NGC_API_KEY:
    print("❌ Need NGC_API_KEY from https://org.ngc.nvidia.com/setup/api-key")
else:
    # Use 8-bit quantized tag for 16GB VRAM
    cmd = [
        "docker", "run", "-d",
        "--name", "flux-kontext",
        "--gpus", "all",
        "-e", f"NGC_API_KEY={NGC_API_KEY}",
        "-e", f"HF_TOKEN={HF_TOKEN}",
        "-p", "8000:8000",
        "-v", "/root/.cache/nim:/opt/nim/.cache",
        "nvcr.io/nim/black-forest-labs/flux.1-kontext-dev:latest"
    ]
    
    print("🐳 Starting NIM container (pulls ~12GB first time)...")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        container_id = result.stdout.strip()
        print(f"✅ Container started: {container_id[:12]}")
    else:
        print(f"❌ Failed: {result.stderr}")
        # Try without quantization if 8-bit fails
        print("🔄 Trying default tag...")
        cmd[-1] = "nvcr.io/nim/black-forest-labs/flux.1-kontext-dev:latest"
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode == 0:
            print(f"✅ Started: {result.stdout.strip()[:12]}")
        else:
            print(f"❌ {result.stderr}")

In [ ]:
# @title 4️⃣ Wait for model to load (2-5 min first run)
import time, requests

print("⏳ Waiting for NIM to be ready...")
for i in range(60):
    try:
        r = requests.get("http://localhost:8000/v1/health/live", timeout=5)
        if r.status_code == 200:
            print(f"✅ NIM ready! (after {i*5}s)")
            break
    except:
        pass
    time.sleep(5)
    if i % 6 == 0:
        print(f"  ... still loading ({i*5}s)")
else:
    print("❌ Timeout - check docker logs")
    !docker logs flux-kontext --tail 50

In [ ]:
# @title 5️⃣ Start ngrok tunnel
if NGROK_TOKEN:
    import subprocess, json
    
    # Start ngrok in background
    tunnel = subprocess.Popen(["ngrok", "http", "8000", "--log", "stdout"], 
                              stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    time.sleep(3)
    
    # Get public URL
    try:
        resp = requests.get("http://localhost:4040/api/tunnels")
        tunnels = resp.json()["tunnels"]
        for t in tunnels:
            if t["proto"] == "https":
                public_url = t["public_url"]
                infer_url = f"{public_url}/v1/infer"
                print(f"\n🌐 Public Inference URL:")
                print(f"   {infer_url}")
                print(f"\n📋 Add to your .env:")
                print(f"   FLUX_INFERENCE_URL={infer_url}")
                break
    except Exception as e:
        print(f"❌ Tunnel error: {e}")
        tunnel.terminate()
else:
    print("⚠️ Set ngrok token in step 2 to get public URL")

In [ ]:
# @title 6️⃣ Test inference
import base64, requests, json

# Test with a simple prompt (no image = text-to-image)
test_payload = {
    "prompt": "A beautiful landscape with mountains and lake, photorealistic",
    "aspect_ratio": "16:9",
    "steps": 20,
    "cfg_scale": 3.5,
    "seed": 42
}

try:
    r = requests.post("http://localhost:8000/v1/infer", 
                      json=test_payload, timeout=120)
    if r.status_code == 200:
        result = r.json()
        print("✅ Inference successful!")
        print(f"Response keys: {list(result.keys())}")
        if 'artifacts' in result:
            print(f"Generated {len(result['artifacts'])} image(s)")
    else:
        print(f"❌ Error {r.status_code}: {r.text}")
except Exception as e:
    print(f"❌ Test failed: {e}")